## 1. import

In [1]:
import numpy as np
import pandas as pd

## 2. config

In [2]:
class CFG:
    SEED = 42
    N_SPLITS = 5
    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    NPY_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/"

## 3. models dict

In [3]:
# ==============================
# Load OOF / PRED
# ==============================
model_dict = {
    "cat_strict_baseline_bias": (
        np.load(CFG.NPY_PATH + "oof_catboost_strict_baseline_bias_proba.npy"),
        np.load(CFG.NPY_PATH + "pred_catboost_strict_baseline_bias_proba.npy"),
    ),
    "cat_strict_baseline_bias_wide": (
        np.load(CFG.NPY_PATH + "oof_catboost_strict_baseline_bias_wide_proba.npy"),
        np.load(CFG.NPY_PATH + "pred_catboost_strict_baseline_bias_wide_proba.npy"),
    ),
    "cat_digit_min_bias": (
        np.load(CFG.NPY_PATH + "oof_catboost_digit_min_bias_proba.npy"),
        np.load(CFG.NPY_PATH + "pred_catboost_digit_min_bias_proba.npy"),
    ),
    "cat_log_min_bias": (
        np.load(CFG.NPY_PATH + "oof_catboost_log_min_bias_proba.npy"),
        np.load(CFG.NPY_PATH + "pred_catboost_log_min_bias_proba.npy"),
    ),
    "cat_pairwise_te_bias_from_shared_min_proba": (
        np.load(CFG.NPY_PATH + "oof_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"),
    ),
}

## 4. show correlations

In [4]:
# ==============================
# Helpers
# ==============================
def flatten_multiclass_proba(arr: np.ndarray) -> np.ndarray:
    """
    (n_samples, n_classes) -> 1d flatten
    相関確認用。まずは全クラスまとめて見る。
    """
    if arr.ndim == 1:
        return arr.astype(np.float64)
    return arr.reshape(-1).astype(np.float64)

def rank01_pd(x: np.ndarray) -> np.ndarray:
    s = pd.Series(x)
    r = s.rank(method="average").values.astype(np.float64)
    denom = max(len(r) - 1, 1)
    return (r - 1.0) / denom

# ==============================
# Build DataFrames
# ==============================
names = list(model_dict.keys())

oof_flat = [flatten_multiclass_proba(model_dict[k][0]) for k in names]
pred_flat = [flatten_multiclass_proba(model_dict[k][1]) for k in names]

oof_lens = [len(x) for x in oof_flat]
pred_lens = [len(x) for x in pred_flat]

if len(set(oof_lens)) != 1:
    raise ValueError(f"OOF lengths mismatch: {dict(zip(names, oof_lens))}")
if len(set(pred_lens)) != 1:
    raise ValueError(f"PRED lengths mismatch: {dict(zip(names, pred_lens))}")

df_oof = pd.DataFrame(np.stack(oof_flat, axis=1), columns=names)
df_pred = pd.DataFrame(np.stack(pred_flat, axis=1), columns=names)

print("df_oof shape :", df_oof.shape)
print("df_pred shape:", df_pred.shape)

# ==============================
# Raw correlation
# ==============================
print("\n=== OOF Pearson corr (raw) ===")
display(df_oof.corr())

print("\n=== TEST Pearson corr (raw) ===")
display(df_pred.corr())

# ==============================
# Rank correlation
# ==============================
df_oof_rank = df_oof.apply(lambda col: rank01_pd(col.values), axis=0, result_type="expand")
df_oof_rank.columns = names

df_pred_rank = df_pred.apply(lambda col: rank01_pd(col.values), axis=0, result_type="expand")
df_pred_rank.columns = names

print("\n=== OOF corr (rank -> Pearson) ===")
display(df_oof_rank.corr())

print("\n=== TEST corr (rank -> Pearson) ===")
display(df_pred_rank.corr())

# ==============================
# Focused view
# ==============================
target = "cat_pairwise_te_bias_from_shared_min_proba"
base_models = [model for model in model_dict.keys() if model != target]

print(f"\n=== Focused correlations vs {target} ===")
for bm in base_models:
    print(
        f"{bm:30s} | "
        f"OOF raw={df_oof[target].corr(df_oof[bm]):.6f} | "
        f"TEST raw={df_pred[target].corr(df_pred[bm]):.6f} | "
        f"OOF rank={df_oof_rank[target].corr(df_oof_rank[bm]):.6f} | "
        f"TEST rank={df_pred_rank[target].corr(df_pred_rank[bm]):.6f}"
    )

df_oof shape : (1890000, 5)
df_pred shape: (810000, 5)

=== OOF Pearson corr (raw) ===


,cat_strict_baseline_bias,cat_strict_baseline_bias_wide,cat_digit_min_bias,cat_log_min_bias,cat_pairwise_te_bias_from_shared_min_proba
cat_strict_baseline_bias,1.000000,0.998880,0.998390,0.998878,-0.491660
cat_strict_baseline_bias_wide,0.998880,1.000000,0.998953,0.999956,-0.491161
cat_digit_min_bias,0.998390,0.998953,1.000000,0.998955,-0.491565
cat_log_min_bias,0.998878,0.999956,0.998955,1.000000,-0.491152
cat_pairwise_te_bias_from_shared_min_proba,-0.491660,-0.491161,-0.491565,-0.491152,1.000000



=== TEST Pearson corr (raw) ===


,cat_strict_baseline_bias,cat_strict_baseline_bias_wide,cat_digit_min_bias,cat_log_min_bias,cat_pairwise_te_bias_from_shared_min_proba
cat_strict_baseline_bias,1.000000,0.999206,0.998818,0.999202,-0.492074
cat_strict_baseline_bias_wide,0.999206,1.000000,0.999369,0.999991,-0.491677
cat_digit_min_bias,0.998818,0.999369,1.000000,0.999371,-0.491878
cat_log_min_bias,0.999202,0.999991,0.999371,1.000000,-0.491690
cat_pairwise_te_bias_from_shared_min_proba,-0.492074,-0.491677,-0.491878,-0.491690,1.000000



=== OOF corr (rank -> Pearson) ===


,cat_strict_baseline_bias,cat_strict_baseline_bias_wide,cat_digit_min_bias,cat_log_min_bias,cat_pairwise_te_bias_from_shared_min_proba
cat_strict_baseline_bias,1.000000,0.996560,0.994903,0.996545,-0.450376
cat_strict_baseline_bias_wide,0.996560,1.000000,0.995681,0.999349,-0.444749
cat_digit_min_bias,0.994903,0.995681,1.000000,0.995625,-0.446279
cat_log_min_bias,0.996545,0.999349,0.995625,1.000000,-0.445429
cat_pairwise_te_bias_from_shared_min_proba,-0.450376,-0.444749,-0.446279,-0.445429,1.000000



=== TEST corr (rank -> Pearson) ===


,cat_strict_baseline_bias,cat_strict_baseline_bias_wide,cat_digit_min_bias,cat_log_min_bias,cat_pairwise_te_bias_from_shared_min_proba
cat_strict_baseline_bias,1.000000,0.998150,0.996698,0.998165,-0.451020
cat_strict_baseline_bias_wide,0.998150,1.000000,0.997234,0.999861,-0.445387
cat_digit_min_bias,0.996698,0.997234,1.000000,0.997217,-0.447346
cat_log_min_bias,0.998165,0.999861,0.997217,1.000000,-0.446147
cat_pairwise_te_bias_from_shared_min_proba,-0.451020,-0.445387,-0.447346,-0.446147,1.000000



=== Focused correlations vs cat_pairwise_te_bias_from_shared_min_proba ===
cat_strict_baseline_bias       | OOF raw=-0.491660 | TEST raw=-0.492074 | OOF rank=-0.450376 | TEST rank=-0.451020
cat_strict_baseline_bias_wide  | OOF raw=-0.491161 | TEST raw=-0.491677 | OOF rank=-0.444749 | TEST rank=-0.445387
cat_digit_min_bias             | OOF raw=-0.491565 | TEST raw=-0.491878 | OOF rank=-0.446279 | TEST rank=-0.447346
cat_log_min_bias               | OOF raw=-0.491152 | TEST raw=-0.491690 | OOF rank=-0.445429 | TEST rank=-0.446147
